In [ ]:
%pip install pennylane pennylane-qiskit qiskit qiskit-ibm-runtime

In [ ]:
import torch
import torch.nn as nn
import pennylane as qml
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from kaggle_secrets import UserSecretsClient # type: ignore
IBM_RUNTIME_API_KEY = UserSecretsClient().get_secret("IBM_RUNTIME_API_KEY")

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

QiskitRuntimeService.save_account(
    token=IBM_RUNTIME_API_KEY,
    overwrite=True,
    set_as_default=True,
)

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService(channel="ibm_cloud")

backend = service.least_busy(
    simulator=False,
    operational=True
)

In [ ]:

Re = 100

n_qubits = 4
initial_layers = 1
max_layers = 4

q_device = qml.device("qiskit.remote", wires=n_qubits, backend=backend)

In [ ]:
import time
from datetime import datetime
from pennylane.qnn import TorchLayer

class QuantumLayer(nn.Module):
    _call_count = 0
    _total_time = 0.0

    def __init__(self, n_layers):
        super().__init__()
        weight_shapes = {"weights": (n_layers, n_qubits, 3)}
        qnode = self.make_circuit(n_layers)
        qnode._set_shots(1024)
        self.layer = TorchLayer(qnode, weight_shapes) # type: ignore

    def forward(self, x):
        t0 = time.perf_counter()
        output = self.layer(x)
        t1 = time.perf_counter()

        elapsed = t1 - t0
        QuantumLayer._call_count += 1
        QuantumLayer._total_time += elapsed

        if QuantumLayer._call_count % 2 == 1:
            msg = (
                f"  [QuantumLayer] call #{QuantumLayer._call_count}: "
                f"{elapsed * 1000:.1f}ms wall-clock for {x.shape[0]} samples"
            )
            # Qiskit job timestamps: break down queue wait vs hardware execution
            try:
                job = backend.jobs(limit=1)[0]
                ts = job.metrics().get("timestamps", {})
                created  = ts.get("created")
                running  = ts.get("running")
                finished = ts.get("finished")
                if created and running and finished:
                    fmt = "%Y-%m-%dT%H:%M:%S.%f%z"
                    t_created  = datetime.strptime(created,  fmt)
                    t_running  = datetime.strptime(running,  fmt)
                    t_finished = datetime.strptime(finished, fmt)
                    queue_ms = (t_running  - t_created ).total_seconds() * 1000
                    exec_ms  = (t_finished - t_running ).total_seconds() * 1000
                    msg += f"  |  queue={queue_ms:.0f}ms  exec={exec_ms:.0f}ms"
            except Exception:
                pass
            print(msg)

        return output

    @staticmethod
    def make_circuit(layers: int):
        @qml.qnode(q_device, interface="torch", diff_method="parameter-shift")
        def circuit(inputs, weights):
            qml.AngleEmbedding(inputs, wires=range(n_qubits), rotation='Y')

            for l in range(layers):
                for q in range(n_qubits):
                    qml.RX(weights[l, q, 0], wires=q)
                    qml.RY(weights[l, q, 1], wires=q)
                    qml.RZ(weights[l, q, 2], wires=q)

                for q in range(n_qubits - 1):
                    qml.CNOT(wires=[q, q + 1])

            return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]
        return circuit


In [1]:
qml.drawer.use_style("black_white") # type: ignore
dummy_inputs = torch.zeros(n_qubits)
dummy_weights = torch.zeros(initial_layers, n_qubits, 3)

fig, ax = qml.draw_mpl(QuantumLayer.make_circuit(initial_layers))(dummy_inputs, dummy_weights)
plt.show()


NameError: name 'qml' is not defined

In [28]:
device_torch = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device_torch)

Device: cpu


In [ ]:
class HybridQPINN(nn.Module):
    _call_count = 0

    def __init__(self, n_layers):
        super().__init__()

        self.embedding = nn.Sequential(
            nn.Linear(2, 32),
            nn.Tanh(),
            nn.Linear(32, n_qubits)
        )

        self.quantum = QuantumLayer(n_layers)

        self.decoder = nn.Sequential(
            nn.Linear(n_qubits, 32),
            nn.Tanh(),
            nn.Linear(32, 2)
        )

    def forward(self, x):
        t0 = time.perf_counter()
        q_in = self.embedding(x)
        t1 = time.perf_counter()
        q_out = self.quantum(q_in)
        t2 = time.perf_counter()
        out = self.decoder(q_out)
        t3 = time.perf_counter()

        HybridQPINN._call_count += 1
        if HybridQPINN._call_count % 2 == 1:
            print(
                f"  [HybridQPINN] call #{HybridQPINN._call_count}: "
                f"embed={( t1 - t0) * 1000:.2f}ms  "
                f"quantum={(t2 - t1) * 1000:.1f}ms  "
                f"decode={(t3 - t2) * 1000:.2f}ms  "
                f"total={(t3 - t0) * 1000:.1f}ms"
            )

        psi = out[:,0:1]
        p = out[:,1:2]

        return psi, p


In [17]:
def velocity(model, coords):
    coords.requires_grad_(True)
    psi,_ = model(coords)

    grad = torch.autograd.grad(
        psi,
        coords,
        grad_outputs=torch.ones_like(psi),
        create_graph=True
    )[0]

    u = grad[:,1:2]
    v = -grad[:,0:1]

    return u,v

In [ ]:
def residual(model, coords):
    coords.requires_grad_(True)

    _psi, p = model(coords)
    u, v = velocity(model, coords)

    grads_u = torch.autograd.grad(u, coords, torch.ones_like(u), create_graph=True)[0]
    grads_v = torch.autograd.grad(v, coords, torch.ones_like(v), create_graph=True)[0]

    u_x = grads_u[:, 0:1]
    u_y = grads_u[:, 1:2]

    v_x = grads_v[:, 0:1]
    v_y = grads_v[:, 1:2]

    u_xx = torch.autograd.grad(u_x, coords, torch.ones_like(u_x), create_graph=True)[0][
        :, 0:1
    ]
    u_yy = torch.autograd.grad(u_y, coords, torch.ones_like(u_y), create_graph=True)[0][
        :, 1:2
    ]

    v_xx = torch.autograd.grad(v_x, coords, torch.ones_like(v_x), create_graph=True)[0][
        :, 0:1
    ]
    v_yy = torch.autograd.grad(v_y, coords, torch.ones_like(v_y), create_graph=True)[0][
        :, 1:2
    ]

    grad_p = torch.autograd.grad(p, coords, torch.ones_like(p), create_graph=True)[0]

    p_x = grad_p[:, 0:1]
    p_y = grad_p[:, 1:2]

    mx = u * u_x + v * u_y + p_x - (1 / Re) * (u_xx + u_yy)
    my = u * v_x + v * v_y + p_y - (1 / Re) * (v_xx + v_yy)

    return mx, my


def boundary_loss(model):
    pts = torch.rand(2000, 2, device=device_torch)
    u, v = velocity(model, pts)

    x = pts[:, 0:1]
    y = pts[:, 1:2]

    lid = y > 0.99
    lid_loss = torch.mean((u[lid] - 1) ** 2) + torch.mean(v[lid] ** 2)
    wall = (y < 0.01) | (x < 0.01) | (x > 0.99)
    wall_loss = torch.mean(u[wall] ** 2) + torch.mean(v[wall] ** 2)

    return lid_loss + wall_loss


In [ ]:
def adaptive_sampling(model, n_points=2000):
    # Reduced candidates: 10 000 → 2 000 to avoid running the quantum circuit
    # on a massive batch just for importance-sampling weights.
    candidate = torch.rand(2000, 2, device=device_torch)
    mx, my = residual(model, candidate)
    res = (mx**2 + my**2).detach().flatten()
    prob = res / res.sum()
    idx = torch.multinomial(prob, n_points, replacement=True)

    return candidate[idx]


In [ ]:
model = HybridQPINN(initial_layers).to(device_torch)

def count_parameters(model):

    total_params = sum(p.numel() for p in model.parameters())

    trainable_params = sum(
        p.numel() for p in model.parameters() if p.requires_grad
    )

    print("Total parameters:", total_params)
    print("Trainable parameters:", trainable_params)

    return total_params, trainable_params

count_parameters(model)


Total parameters: 466
Trainable parameters: 466


(466, 466)

In [ ]:
import time

layers = initial_layers
epoch_times = []

QuantumLayer._call_count = 0
QuantumLayer._total_time = 0.0
HybridQPINN._call_count = 0

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(25):
    ep_start = time.perf_counter()

    pts = adaptive_sampling(model, 2000)
    mx, my = residual(model, pts)
    phys_loss = torch.mean(mx**2) + torch.mean(my**2)
    bc = boundary_loss(model)
    loss = phys_loss + bc

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    epoch_time = time.perf_counter() - ep_start
    epoch_times.append(epoch_time)

    if epoch % 2 == 0:
        print(f"Epoch {epoch} | Loss {loss:.6f}")

    # depth scaling
    # if epoch > 0 and epoch % 2000 == 0 and initial_layers < max_layers:
    #     layers += 1
    #     print("Increasing quantum depth to", layers)

    #     model = HybridQPINN(layers).to(device_torch)
    #     QuantumLayer._call_count = 0
    #     QuantumLayer._total_time = 0.0
    #     HybridQPINN._call_count = 0
    #     optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


In [ ]:
import time

grid = 60
x = np.linspace(0, 1, grid)
y = np.linspace(0, 1, grid)
X, Y = np.meshgrid(x, y)
coords = torch.tensor(
    np.vstack([X.flatten(), Y.flatten()]).T, dtype=torch.float32
).to(device_torch)

n_points = coords.shape[0]

def qiskit_queue_ms():
    """Return queue-wait ms from the most recent Qiskit job, or 0.0 on failure."""
    try:
        ts = backend.jobs(limit=1)[0].metrics().get("timestamps", {})
        created = ts.get("created")
        running = ts.get("running")
        if created and running:
            fmt = "%Y-%m-%dT%H:%M:%S.%f%z"
            return (datetime.strptime(running, fmt) - datetime.strptime(created, fmt)).total_seconds() * 1000
    except Exception:
        pass
    return 0.0

# --- prediction timing ---
with torch.no_grad():
    t_pred_start = time.perf_counter()
    u, v = velocity(model, coords)
    t_pred_end = time.perf_counter()

wall_pred_time = t_pred_end - t_pred_start
queue_pred_ms  = qiskit_queue_ms()
net_pred_time  = wall_pred_time - queue_pred_ms / 1000

print(
    f"Prediction — wall: {wall_pred_time * 1000:.1f}ms  "
    f"queue: {queue_pred_ms:.1f}ms  "
    f"net: {net_pred_time * 1000:.1f}ms  "
    f"per point (net): {net_pred_time / n_points * 1e6:.2f}µs  "
    f"points: {n_points}"
)

U = u.detach().cpu().numpy().reshape(grid, grid)
V = v.detach().cpu().numpy().reshape(grid, grid)

plt.figure(figsize=(6, 6))
plt.streamplot(X, Y, U, V, density=2)
plt.xlabel("x")
plt.ylabel("y")
plt.title(f"Lid-Driven Cavity Streamlines (Re={Re})")
plt.tight_layout()
plt.show()


In [ ]:
# ── Ghia et al. (1982) benchmark data for Re=100 ──────────────────────────────
# u-velocity along vertical centreline x = 0.5
ghia_y = np.array([0.0000, 0.0547, 0.0625, 0.0703, 0.1016, 0.1719, 0.2813,
                    0.4531, 0.5000, 0.6172, 0.7344, 0.8516, 0.9531, 0.9609,
                    0.9688, 0.9766, 1.0000])
ghia_u = np.array([0.0000, -0.0372, -0.0419, -0.0478, -0.0643, -0.1015, -0.1566,
                   -0.2109, -0.2058, -0.1364,  0.0033,  0.2315,  0.6872,  0.7372,
                    0.7887,  0.8412,  1.0000])

# v-velocity along horizontal centreline y = 0.5
ghia_x = np.array([0.0000, 0.0625, 0.0703, 0.0781, 0.0938, 0.1563, 0.2266,
                    0.2344, 0.5000, 0.8047, 0.8594, 0.9063, 0.9453, 0.9531,
                    0.9609, 0.9688, 1.0000])
ghia_v = np.array([0.0000,  0.0965,  0.1066,  0.1202,  0.1465,  0.1812,  0.1795,
                    0.1750,  0.0545, -0.2273, -0.2008, -0.1674, -0.1091, -0.0941,
                   -0.0783, -0.0588,  0.0000])

# ── model predictions along centrelines ───────────────────────────────────────
with torch.no_grad():
    t0 = time.perf_counter()
    cl_v_pts = torch.tensor(
        np.column_stack([np.full_like(ghia_y, 0.5), ghia_y]), dtype=torch.float32
    ).to(device_torch)
    u_pred, _ = velocity(model, cl_v_pts)
    t1 = time.perf_counter()
    u_queue_ms = qiskit_queue_ms()
    u_pred = u_pred.cpu().numpy().flatten()

    cl_h_pts = torch.tensor(
        np.column_stack([ghia_x, np.full_like(ghia_x, 0.5)]), dtype=torch.float32
    ).to(device_torch)
    _, v_pred = velocity(model, cl_h_pts)
    t2 = time.perf_counter()
    v_queue_ms = qiskit_queue_ms()
    v_pred = v_pred.cpu().numpy().flatten()

u_wall_ms = (t1 - t0) * 1000
v_wall_ms = (t2 - t1) * 1000
print(f"Centreline prediction (wall | queue | net):")
print(f"  u: {u_wall_ms:.1f}ms wall  {u_queue_ms:.1f}ms queue  {u_wall_ms - u_queue_ms:.1f}ms net  ({len(ghia_y)} pts)")
print(f"  v: {v_wall_ms:.1f}ms wall  {v_queue_ms:.1f}ms queue  {v_wall_ms - v_queue_ms:.1f}ms net  ({len(ghia_x)} pts)")

u_rmse = np.sqrt(np.mean((u_pred - ghia_u) ** 2))
v_rmse = np.sqrt(np.mean((v_pred - ghia_v) ** 2))
u_mae  = np.mean(np.abs(u_pred - ghia_u))
v_mae  = np.mean(np.abs(v_pred - ghia_v))

print(f"\nComparison with Ghia et al. (1982) at Re={Re}")
print(f"  u-centreline  RMSE={u_rmse:.4f}  MAE={u_mae:.4f}")
print(f"  v-centreline  RMSE={v_rmse:.4f}  MAE={v_mae:.4f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(ghia_u, ghia_y, "ko-", markersize=5, label="Ghia et al. (1982)")
ax1.plot(u_pred, ghia_y, "r--",               label="Hybrid QPINN")
ax1.set_xlabel("u  velocity")
ax1.set_ylabel("y")
ax1.set_title(f"u along x=0.5  (Re={Re})")
ax1.legend()
ax1.grid(True, linestyle=":")

ax2.plot(ghia_x, ghia_v, "ko-", markersize=5, label="Ghia et al. (1982)")
ax2.plot(ghia_x, v_pred, "r--",               label="Hybrid QPINN")
ax2.set_xlabel("x")
ax2.set_ylabel("v  velocity")
ax2.set_title(f"v along y=0.5  (Re={Re})")
ax2.legend()

ax2.grid(True, linestyle=":")

plt.show()
plt.tight_layout()